In [ ]:
from jupytergis import GISDocument

doc = GISDocument()

doc.add_raster_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Satellite",
    attribution="Google",
    opacity=0.6,
)

doc

In [ ]:
doc = GISDocument(latitude=20.718, longitude=129.940, zoom=4)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="Open Street Map",
    attribution="OpenStreetMap",
    opacity=0.6,
)
doc

In [ ]:
doc = GISDocument(latitude=20.718, longitude=129.940, zoom=4)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="Open Street Map",
    attribution="OpenStreetMap",
    opacity=0.6,
)

doc.add_geojson_layer(path="./eq.geojson")

doc

In [ ]:
doc = GISDocument()

doc.add_heatmap_layer(path="./eq.geojson", feature="mag")

doc

## Tiler tests

### Setup mock data

In [ ]:
import numpy as np
import rioxarray  # noqa: F401
import xarray as xra

NPIXELS_Y = 100
NPIXELS_X = 100
MIN_X = -180
MAX_X = 180
MIN_Y = -90
MAX_Y = 90
RES_X = (MAX_X - MIN_X) / NPIXELS_X
RES_Y = (MAX_Y - MIN_Y) / NPIXELS_Y

# Coordinates at pixel centers
y_coords = np.linspace(MAX_Y - RES_Y / 2, MIN_Y + RES_Y / 2, NPIXELS_Y)
x_coords = np.linspace(MIN_X + RES_X / 2, MAX_X - RES_X / 2, NPIXELS_X)
x_grid, y_grid = np.meshgrid(x_coords, y_coords)

data = ((x_grid - x_grid.min()) + (y_grid - y_grid.min())) / 2  # Diagonal gradient
data = data / data.max()  # Normalize to 0-1

da = xra.DataArray(
    data,
    dims=["latitude", "longitude"],
    coords={
        "latitude": y_coords,
        "longitude": x_coords,
    },
)
da.rio.write_crs("EPSG:4326", inplace=True)

doc = GISDocument()

await doc.add_data_array_layer(da, colormap_range=(0, 1))

doc